# Qwen geometry + generation calibration (полный Colab run)

Этот notebook запускает **полный калибровочный прогон без `limit` и без `case_name`** для `Qwen/Qwen2.5-1.5B` через `C:/Users/Kharki/Desktop/ABPT/scripts/run_qwen_geometry_generation_calibration.py`.

Что делает прогон:
- один geometry forward на слоях `18..27` для каждого anchor-кейса
- полная base generation
- полная anchor-biased generation
- keyword/constraint analysis
- запись итогов в JSON и Markdown

Это именно **full run harness для Google Colab GPU**, а не chunked smoke-test.


> Рекомендуемый runtime: **GPU (T4 / L4 / A100)**.
>
> На CPU запуск тоже возможен, но будет очень медленным.
>
> По умолчанию notebook гоняет **все кейсы целиком**.


In [ ]:
!nvidia-smi || true
!python --version
!pip install -q torch transformers einops pytest numpy


In [ ]:
%cd /content
import os

if not os.path.exists('/content/ABPT'):
    !git clone https://github.com/kharkilirov1/Anchor-engine.git ABPT
%cd /content/ABPT
!git fetch --all
!git checkout main
!git pull --ff-only origin main


In [ ]:
import json
import shlex
import subprocess
from pathlib import Path

import torch

from src.data.qwen_anchor_geometry_cases import make_qwen_anchor_geometry_cases

MODEL_NAME = 'Qwen/Qwen2.5-1.5B'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_LENGTH = 128
MAX_NEW_TOKENS = 120
SEED = 7

OUTPUT_JSON = '/content/ABPT/archive/qwen_geometry_generation_calibration.json'
OUTPUT_MD = '/content/ABPT/docs/research/qwen_geometry_generation_calibration.md'

cases = make_qwen_anchor_geometry_cases()
print('device =', DEVICE)
print('n_cases =', len(cases))
print('case names =', [case.name for case in cases])
print('output_json =', OUTPUT_JSON)
print('output_md =', OUTPUT_MD)


In [ ]:
for case in make_qwen_anchor_geometry_cases():
    print(f'[{case.anchor_class}] {case.name}')
    print(' anchor_group:', case.anchor_group)
    print(' anchor_text :', case.anchor_text)
    print(' prompt      :', case.prompt)
    print()


In [ ]:
cmd = [
    'python',
    '-u',
    'scripts/run_qwen_geometry_generation_calibration.py',
    '--model', MODEL_NAME,
    '--device', DEVICE,
    '--max_length', str(MAX_LENGTH),
    '--max_new_tokens', str(MAX_NEW_TOKENS),
    '--seed', str(SEED),
    '--output_json', OUTPUT_JSON,
    '--output_md', OUTPUT_MD,
]
print('Running full calibration command:')
print(' '.join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, check=True)


In [ ]:
payload = json.loads(Path(OUTPUT_JSON).read_text(encoding='utf-8'))
print('metadata:')
for key, value in payload['metadata'].items():
    print(f'  {key}: {value}')

print()
print('clean-base calibration summary:')
print('excluded_base_degenerate_case_names:', payload['calibration']['excluded_base_degenerate_case_names'])
for cluster, stats in payload['calibration']['by_cluster_clean_base'].items():
    print(cluster, stats)
print('threshold_candidates:', payload['calibration']['threshold_candidates'])

print()
print('degenerate-base rescue summary:')
for cluster, stats in payload['calibration']['by_cluster_degenerate_base'].items():
    print(cluster, stats)

print()
print('policy simulation:')
for subset_name, subset_stats in payload['policy_simulation'].items():
    print(f'[{subset_name}]')
    for policy_name, policy_stats in subset_stats.items():
        print(policy_name, policy_stats)
    print()

print('per-case summary:')
for case in payload['cases']:
    print(
        f"{case['name']}: cluster={case['anchor_cluster']} | "
        f"r1@24={case['r1_at_24']} | d26->27={case['delta_l26_l27']} | "
        f"base_c={case['base_analysis']['constraint_score']} | "
        f"anchor_c={case['anchor_analysis']['constraint_score']} | "
        f"delta={case['constraint_delta']} | "
        f"base_deg={case['base_degenerate']} | "
        f"drift={case['anchor_analysis']['drift_detected']}"
    )


In [ ]:
report_text = Path(OUTPUT_MD).read_text(encoding='utf-8')
print(report_text)


## Tree-guided benchmark with geometry routing

Этот блок **включает новый geometry-routed метод** внутри `tree_guided_generate()` и сравнивает его со старым tree-guided path на том же calibration prompt set.

Сравниваются два режима:
- legacy: `use_geometry_routing=False`
- geometry-routed: `use_geometry_routing=True`

Артефакты benchmark сохраняются отдельно в JSON и Markdown.


In [ ]:
from collections import Counter
from dataclasses import replace
from datetime import UTC, datetime

from src.model.config import TOY_CONFIG
from src.model.qwen_anchor_overlay import QwenAnchorOverlay
from scripts.run_qwen_geometry_generation_calibration import KEYWORD_MAP, compute_constraint_analysis

TREE_BENCHMARK_MAX_NEW_TOKENS = 120
TREE_BENCHMARK_MAX_LENGTH = 256
TREE_BENCHMARK_TEMPERATURE = 0.7
TREE_BENCHMARK_CHUNK_SIZE = 20
TREE_BENCHMARK_TREE_CHECK_INTERVAL = 10
TREE_BENCHMARK_MAX_REVISIONS = 3

OUTPUT_TREE_BENCH_JSON = '/content/ABPT/archive/qwen_tree_guided_geometry_benchmark.json'
OUTPUT_TREE_BENCH_MD = '/content/ABPT/docs/research/qwen_tree_guided_geometry_benchmark.md'

TREE_CFG = replace(
    TOY_CONFIG,
    anchor_threshold=0.10,
    anchor_revision_threshold=0.35,
    anchor_contradiction_threshold=0.20,
    anchor_dead_end_threshold=0.50,
)

tree_overlay = QwenAnchorOverlay.from_pretrained(
    model_name=MODEL_NAME,
    cfg=TREE_CFG,
    device=DEVICE,
    torch_dtype=torch.float16 if DEVICE.startswith('cuda') else None,
    low_cpu_mem_usage=True,
)
tree_overlay.eval()

print('tree benchmark model loaded on', DEVICE)
print('output_tree_json =', OUTPUT_TREE_BENCH_JSON)
print('output_tree_md =', OUTPUT_TREE_BENCH_MD)


In [ ]:
def tree_case_analysis(case, text):
    keyword_spec = KEYWORD_MAP.get(case.anchor_group)
    if keyword_spec is None:
        return {
            'constraint_score': None,
            'constraint_satisfied': False,
            'degenerate_output': False,
            'drift_detected': False,
        }
    return compute_constraint_analysis(text, keyword_spec)


tree_benchmark_rows = []
for case in make_qwen_anchor_geometry_cases():
    torch.manual_seed(SEED)
    legacy = tree_overlay.tree_guided_generate(
        prompt=case.prompt,
        max_new_tokens=TREE_BENCHMARK_MAX_NEW_TOKENS,
        max_length=TREE_BENCHMARK_MAX_LENGTH,
        temperature=TREE_BENCHMARK_TEMPERATURE,
        chunk_size=TREE_BENCHMARK_CHUNK_SIZE,
        tree_check_interval=TREE_BENCHMARK_TREE_CHECK_INTERVAL,
        max_revisions=TREE_BENCHMARK_MAX_REVISIONS,
        use_geometry_routing=False,
    )

    torch.manual_seed(SEED)
    routed = tree_overlay.tree_guided_generate(
        prompt=case.prompt,
        max_new_tokens=TREE_BENCHMARK_MAX_NEW_TOKENS,
        max_length=TREE_BENCHMARK_MAX_LENGTH,
        temperature=TREE_BENCHMARK_TEMPERATURE,
        chunk_size=TREE_BENCHMARK_CHUNK_SIZE,
        tree_check_interval=TREE_BENCHMARK_TREE_CHECK_INTERVAL,
        max_revisions=TREE_BENCHMARK_MAX_REVISIONS,
        use_geometry_routing=True,
    )

    legacy_analysis = tree_case_analysis(case, legacy['continuation_text'])
    routed_analysis = tree_case_analysis(case, routed['continuation_text'])
    row = {
        'name': case.name,
        'anchor_group': case.anchor_group,
        'prompt': case.prompt,
        'legacy_generation_mode': legacy.get('generation_mode', 'tree_guided'),
        'legacy_constraint_score': legacy_analysis['constraint_score'],
        'legacy_constraint_satisfied': legacy_analysis['constraint_satisfied'],
        'legacy_revision_count': legacy.get('revision_count', 0),
        'legacy_continuation_text': legacy['continuation_text'],
        'routed_generation_mode': routed.get('generation_mode', 'unknown'),
        'routed_cluster': routed.get('geometry_route', {}).get('cluster', 'unknown'),
        'routed_route': routed.get('geometry_route', {}).get('route', 'unknown'),
        'routed_constraint_score': routed_analysis['constraint_score'],
        'routed_constraint_satisfied': routed_analysis['constraint_satisfied'],
        'routed_revision_count': routed.get('revision_count', 0),
        'routed_continuation_text': routed['continuation_text'],
        'delta': float((routed_analysis['constraint_score'] or 0.0) - (legacy_analysis['constraint_score'] or 0.0)),
    }
    tree_benchmark_rows.append(row)
    print(
        f"{case.name}: legacy={row['legacy_constraint_score']} | "
        f"routed={row['routed_constraint_score']} | "
        f"delta={row['delta']} | mode={row['routed_generation_mode']} | "
        f"cluster={row['routed_cluster']}"
    )
    print('prompt:')
    print(row['prompt'])
    print('legacy continuation:')
    print(row['legacy_continuation_text'])
    print('routed continuation:')
    print(row['routed_continuation_text'])
    print('-' * 100)


In [ ]:
legacy_scores = [float(row['legacy_constraint_score'] or 0.0) for row in tree_benchmark_rows]
routed_scores = [float(row['routed_constraint_score'] or 0.0) for row in tree_benchmark_rows]
mode_counts = Counter(row['routed_generation_mode'] for row in tree_benchmark_rows)
cluster_counts = Counter(row['routed_cluster'] for row in tree_benchmark_rows)
route_counts = Counter(row['routed_route'] for row in tree_benchmark_rows)

wins = sum(1 for row in tree_benchmark_rows if row['delta'] > 0)
losses = sum(1 for row in tree_benchmark_rows if row['delta'] < 0)
ties = sum(1 for row in tree_benchmark_rows if row['delta'] == 0)

summary = {
    'n_cases': len(tree_benchmark_rows),
    'legacy_mean_constraint_score': float(sum(legacy_scores) / len(legacy_scores)) if legacy_scores else None,
    'routed_mean_constraint_score': float(sum(routed_scores) / len(routed_scores)) if routed_scores else None,
    'delta_vs_legacy': float((sum(routed_scores) - sum(legacy_scores)) / len(routed_scores)) if routed_scores else None,
    'wins': wins,
    'losses': losses,
    'ties': ties,
    'mode_counts': dict(mode_counts),
    'cluster_counts': dict(cluster_counts),
    'route_counts': dict(route_counts),
}

payload_tree_bench = {
    'metadata': {
        'created_at_utc': datetime.now(UTC).isoformat(),
        'model_name': MODEL_NAME,
        'device': DEVICE,
        'max_new_tokens': TREE_BENCHMARK_MAX_NEW_TOKENS,
        'max_length': TREE_BENCHMARK_MAX_LENGTH,
        'temperature': TREE_BENCHMARK_TEMPERATURE,
        'chunk_size': TREE_BENCHMARK_CHUNK_SIZE,
        'tree_check_interval': TREE_BENCHMARK_TREE_CHECK_INTERVAL,
        'max_revisions': TREE_BENCHMARK_MAX_REVISIONS,
        'seed': SEED,
    },
    'results': tree_benchmark_rows,
    'summary': summary,
}

report_lines = [
    '# Qwen Tree-Guided Geometry Routing Benchmark',
    '',
    '## Summary',
    '',
    f"- Date: {datetime.now(UTC).strftime('%Y-%m-%d %H:%M UTC')}",
    f"- Model: `{MODEL_NAME}`",
    f"- Device: `{DEVICE}`",
    f"- n_cases: `{summary['n_cases']}`",
    f"- legacy_mean_constraint_score: `{summary['legacy_mean_constraint_score']:.3f}`",
    f"- routed_mean_constraint_score: `{summary['routed_mean_constraint_score']:.3f}`",
    f"- delta_vs_legacy: `{summary['delta_vs_legacy']:.3f}`",
    f"- wins: `{summary['wins']}`",
    f"- losses: `{summary['losses']}`",
    f"- ties: `{summary['ties']}`",
    f"- mode_counts: `{summary['mode_counts']}`",
    f"- cluster_counts: `{summary['cluster_counts']}`",
    f"- route_counts: `{summary['route_counts']}`",
    '',
    '## Per-case table',
    '',
    '| name | legacy_score | routed_score | delta | routed_mode | routed_cluster | routed_route |',
    '| --- | ---: | ---: | ---: | --- | --- | --- |',
]
for row in tree_benchmark_rows:
    report_lines.append(
        '| ' + ' | '.join([
            row['name'],
            f"{float(row['legacy_constraint_score'] or 0.0):.3f}",
            f"{float(row['routed_constraint_score'] or 0.0):.3f}",
            f"{float(row['delta']):.3f}",
            row['routed_generation_mode'],
            row['routed_cluster'],
            row['routed_route'],
        ]) + ' |'
    )

report_lines.extend(['', '## Per-case generations', ''])
for row in tree_benchmark_rows:
    report_lines.extend([
        f"### {row['name']}",
        '',
        f"- routed_mode: `{row['routed_generation_mode']}`",
        f"- routed_cluster: `{row['routed_cluster']}`",
        f"- routed_route: `{row['routed_route']}`",
        f"- legacy_score: `{float(row['legacy_constraint_score'] or 0.0):.3f}`",
        f"- routed_score: `{float(row['routed_constraint_score'] or 0.0):.3f}`",
        f"- delta: `{float(row['delta']):.3f}`",
        '',
        '**Prompt**',
        '',
        '```text',
        row['prompt'],
        '```',
        '',
        '**Legacy continuation**',
        '',
        '```text',
        row['legacy_continuation_text'],
        '```',
        '',
        '**Geometry-routed continuation**',
        '',
        '```text',
        row['routed_continuation_text'],
        '```',
        '',
    ])
report_text = '\n'.join(report_lines) + '\n'

Path(OUTPUT_TREE_BENCH_JSON).write_text(json.dumps(payload_tree_bench, ensure_ascii=False, indent=2), encoding='utf-8')
Path(OUTPUT_TREE_BENCH_MD).write_text(report_text, encoding='utf-8')

print('tree benchmark summary:')
print(summary)
print()
print(report_text)


In [ ]:
# Optional: download artifacts from Colab runtime
# from google.colab import files
# files.download(OUTPUT_JSON)
# files.download(OUTPUT_MD)
# files.download(OUTPUT_TREE_BENCH_JSON)
# files.download(OUTPUT_TREE_BENCH_MD)
